<a href="https://colab.research.google.com/github/MesyAnnariaSibarani/Pemrograman-Dasar/blob/main/Tugas_2_Modul_KPI_Pembangkit_Listrik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# SOAL 1: DEFINISI FUNGSI KPI & STATUS OPERASI
# ==============================================================================

def heat_rate(coal_t, energy_mwh, gcv_kcal_kg=4400):
    """Menghitung laju panas (kcal/kWh) dari batubara (t) dan energi (MWh)."""
    if energy_mwh <= 0:
        raise ValueError("energy_mwh harus lebih besar dari 0")
    return coal_t * gcv_kcal_kg / energy_mwh


def thermal_efficiency(heat_rate_kcal_kwh):
    """Menghitung efisiensi termal (%) dari laju panas (1 kWh = 860 kcal)."""
    if heat_rate_kcal_kwh <= 0:
        raise ValueError("heat_rate_kcal_kwh harus lebih besar dari 0")
    return 860 / heat_rate_kcal_kwh * 100


def capacity_factor(energy_mwh, installed_mw=330, hours=24):
    """Menghitung faktor kapasitas (%) untuk periode operasi tertentu."""
    if installed_mw <= 0 or hours <= 0:
        raise ValueError("installed_mw dan hours harus lebih besar dari 0")
    return energy_mwh / (installed_mw * hours) * 100


def specific_coal_consumption(coal_t, energy_mwh):
    """Menghitung konsumsi batubara spesifik (t/MWh)."""
    if energy_mwh <= 0:
        raise ValueError("energy_mwh harus lebih besar dari 0")
    return coal_t / energy_mwh


def day_status(efficiency, run_hours):
    """Menentukan status harian: NOT FULL (<24 jam), LOW EFFICIENCY (<36.3%), atau NORMAL."""
    if run_hours < 24:
        return "NOT FULL"
    if efficiency < 36.3:
        return "LOW EFFICIENCY"
    return "NORMAL"


def lowest_days(values, n=3):
    """Mengembalikan daftar nomor hari (mulai dari 1) untuk n nilai efisiensi terendah."""
    temp = list(values)  # Menyalin list agar data asli tidak berubah
    result = []
    for _ in range(n):
        min_val = min(temp)
        idx = temp.index(min_val)
        result.append(idx + 1)
        temp[idx] = float('inf')  # Tandai nilai terkecil yang sudah ditemukan
    return result

In [3]:
# ==============================================================================
# SOAL 5: PENGUJIAN (ASSERT) DENGAN PERHITUNGAN MANUAL
# ==============================================================================

# 1. Manual: 1 t * 4400 kcal/kg / 2 MWh = 2200 kcal/kWh
assert heat_rate(1, 2) == 2200

# 2. Manual: 860 / 2150 * 100 = 40.0 %
assert round(thermal_efficiency(2150), 2) == 40.0

# 3. Manual: 7920 MWh / (330 MW * 24 jam) * 100 = 100.0 %
assert round(capacity_factor(7920, 330, 24), 1) == 100.0

# 4. Manual: 100 t / 200 MWh = 0.5 t/MWh
assert specific_coal_consumption(100, 200) == 0.5

# 5. Manual uji batas status
assert day_status(35.0, 18) == "NOT FULL"        # Jam < 24 diprioritaskan
assert day_status(36.1, 24) == "LOW EFFICIENCY"  # Jam 24, efisiensi < 36.3
assert day_status(36.3, 24) == "NORMAL"          # Tepat 36.3 masuk NORMAL

print("Semua pengujian assert berhasil lolos!\n")

Semua pengujian assert berhasil lolos!



In [9]:
# ==============================================================================
# SOAL 2 & 3: PROSEDUR TABEL HARIAN DAN RINGKASAN KPI
# ==============================================================================

def print_report(unit_name, installed_mw, gcv_kcal_kg, energy_list, coal_list, run_hours_list):
    """Prosedur: mencetak tabel KPI harian serta ringkasan statistik untuk satu unit."""
    print("=" * 68)
    print(f"LAPORAN KPI PEMBANGKIT - {unit_name.upper()} ({installed_mw} MW, GCV {gcv_kcal_kg} kcal/kg)")
    print("=" * 68)
    print(f"{'Day':>3} {'HR kcal':>8} {'Eff %':>7} {'CF %':>6} {'SCC t/MWh':>10}  {'Status'}")
    print("-" * 68)

    efficiencies = []
    status_counts = {"NOT FULL": 0, "LOW EFFICIENCY": 0, "NORMAL": 0}

    for i in range(len(energy_list)):
        hr = heat_rate(coal_list[i], energy_list[i], gcv_kcal_kg=gcv_kcal_kg)
        eff = thermal_efficiency(hr)
        cf = capacity_factor(energy_list[i], installed_mw=installed_mw, hours=24)
        scc = specific_coal_consumption(coal_list[i], energy_list[i])
        status = day_status(eff, run_hours_list[i])

        efficiencies.append(eff)
        status_counts[status] += 1

        print(f"{i + 1:>3} {hr:>8,.0f} {eff:>7.2f} {cf:>6.1f} {scc:>10.3f}  {status}")

    # --- Perhitungan Ringkasan (Soal 3) ---
    avg_eff = sum(efficiencies) / len(efficiencies)
    period_cf = capacity_factor(
        sum(energy_list),
        installed_mw=installed_mw,
        hours=24 * len(energy_list)
    )
    low_3 = lowest_days(efficiencies, n=3)

    print("-" * 68)
    print(f"Efisiensi rata-rata       : {avg_eff:.2f} %")
    print(f"Faktor kapasitas periode  : {period_cf:.1f} %")
    print(f"Jumlah hari per status    : NORMAL = {status_counts['NORMAL']}, "
          f"LOW EFFICIENCY = {status_counts['LOW EFFICIENCY']}, "
          f"NOT FULL = {status_counts['NOT FULL']}")
    print(f"3 hari efisiensi terendah : "
          f"Hari {low_3[0]} ({efficiencies[low_3[0]-1]:.2f}%), "
          f"Hari {low_3[1]} ({efficiencies[low_3[1]-1]:.2f}%), "
          f"Hari {low_3[2]} ({efficiencies[low_3[2]-1]:.2f}%)")
    print("=" * 68 + "\n")

In [7]:
# ==============================================================================
# EKSEKUSI UNIT 1 (30 HARI) & SOAL 4: PENGGUNAAN ULANG UNTUK UNIT 2 (5 HARI)
# ==============================================================================

# --- Data Unit 1 (330 MW, GCV 4400 kcal/kg, 30 Hari) ---
energy_u1 = [
    5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
    6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
    5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062
]
coal_u1 = [
    2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
    3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
    3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240
]
run_hours_u1 = [24] * 30
run_hours_u1[16] = 18  # Hari ke-17 (indeks 16) hanya beroperasi 18 jam

print_report("Unit 1", 330, 4400, energy_u1, coal_u1, run_hours_u1)


# --- Data Unit 2 (600 MW, GCV 5100 kcal/kg, 5 Hari) ---
energy_u2 = [12150, 12080, 11020, 12210, 12095]
coal_u2 = [5240, 5225, 4790, 5260, 5230]
run_hours_u2 = [24] * 5  # Asumsi beroperasi penuh 24 jam per hari

# Memanggil prosedur yang sama tanpa menyalin rumus
print_report("Unit 2", 600, 5100, energy_u2, coal_u2, run_hours_u2)

LAPORAN KPI PEMBANGKIT - UNIT 1 (330 MW, GCV 4400 kcal/kg)
Day  HR kcal   Eff %   CF %  SCC t/MWh  Status
--------------------------------------------------------------------
  1    2,366   36.35   70.4      0.538  NORMAL
  2    2,351   36.58   77.0      0.534  NORMAL
  3    2,353   36.55   76.7      0.535  NORMAL
  4    2,356   36.51   77.0      0.535  NORMAL
  5    2,354   36.54   76.7      0.535  NORMAL
  6    2,350   36.59   76.7      0.534  NORMAL
  7    2,367   36.34   70.3      0.538  NORMAL
  8    2,364   36.38   70.1      0.537  NORMAL
  9    2,350   36.60   77.0      0.534  NORMAL
 10    2,357   36.49   76.8      0.536  NORMAL
 11    2,348   36.63   77.0      0.534  NORMAL
 12    2,348   36.63   76.9      0.534  NORMAL
 13    2,353   36.55   76.6      0.535  NORMAL
 14    2,383   36.10   69.9      0.541  LOW EFFICIENCY
 15    2,379   36.15   70.3      0.541  LOW EFFICIENCY
 16    2,350   36.59   77.0      0.534  NORMAL
 17    2,389   36.00   45.4      0.543  NOT FULL
 18    2